# Per-Residue Structural Variance (PRSV)

This Colab notebook runs `per-residue-structural-variance.R` from the `cianfrocco-lab/nvangos` repository. It installs the R dependencies, loads the PRSV function, reads aligned tubulin PDB files, and writes PRSV outputs.

Use the interactive widget panel to choose the alpha- and beta-tubulin chain IDs separately for the base PDB and comparison PDB.

## 1. Clone the repository

In [ ]:
!git clone --depth 1 https://github.com/cianfrocco-lab/nvangos.git
%cd nvangos

## 2. Install R dependencies

In [ ]:
%%bash
Rscript - <<'RSCRIPT'
packages <- c("tidyverse", "bio3d", "ggpubr")
installed <- rownames(installed.packages())
missing <- setdiff(packages, installed)
if (length(missing) > 0) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
RSCRIPT

## 3. Upload input PDB files

Upload one base PDB file and one comparison PDB file from your computer.

In [ ]:
# Upload file 1: base PDB.
from google.colab import files

uploaded_base = files.upload()
if len(uploaded_base) != 1:
    raise ValueError("Please upload exactly one base PDB file in this cell.")

BASE_PDB_FILE = next(iter(uploaded_base.keys()))
BASE_PDB_FILE

In [ ]:
# Upload file 2: comparison PDB.
from google.colab import files

uploaded_comparison = files.upload()
if len(uploaded_comparison) != 1:
    raise ValueError("Please upload exactly one comparison PDB file in this cell.")

COMPARISON_PDB_FILE = next(iter(uploaded_comparison.keys()))
COMPARISON_PDB_FILE

## 4. Choose files and chain IDs

In [ ]:
import ipywidgets as widgets
from IPython.display import display

if "BASE_PDB_FILE" not in globals() or "COMPARISON_PDB_FILE" not in globals():
    raise RuntimeError("Run both upload cells before creating the chain-selection widgets.")

def chain_ids(path):
    chains = []
    seen = set()
    with open(path) as handle:
        for line in handle:
            if line.startswith(("ATOM", "HETATM")):
                chain = line[21].strip() if len(line) > 21 else ""
                label = chain if chain else "(blank)"
                if label not in seen:
                    seen.add(label)
                    chains.append(label)
    if not chains:
        raise ValueError(f"No ATOM/HETATM chain IDs found in {path}")
    return chains

def label_to_chain(label):
    return "" if label == "(blank)" else label

base_alpha_chain = widgets.Dropdown(description="Base alpha chain:", style={"description_width": "initial"})
base_beta_chain = widgets.Dropdown(description="Base beta chain:", style={"description_width": "initial"})
comparison_alpha_chain = widgets.Dropdown(description="Comparison alpha chain:", style={"description_width": "initial"})
comparison_beta_chain = widgets.Dropdown(description="Comparison beta chain:", style={"description_width": "initial"})
comparison_name = widgets.Text(value="base-vs-comparison", description="Output prefix:", style={"description_width": "initial"})

def set_default_chain(dropdown, options, preferred):
    dropdown.options = options
    dropdown.value = preferred if preferred in options else options[0]

def update_base_chains(*args):
    options = chain_ids(BASE_PDB_FILE)
    set_default_chain(base_alpha_chain, options, "A")
    set_default_chain(base_beta_chain, options, "B")

def update_comparison_chains(*args):
    options = chain_ids(COMPARISON_PDB_FILE)
    set_default_chain(comparison_alpha_chain, options, "A")
    set_default_chain(comparison_beta_chain, options, "B")

update_base_chains()
update_comparison_chains()

display(widgets.VBox([
    widgets.HTML(f"<b>Base PDB:</b> {BASE_PDB_FILE}"),
    widgets.HTML(f"<b>Comparison PDB:</b> {COMPARISON_PDB_FILE}"),
    widgets.HBox([base_alpha_chain, base_beta_chain]),
    widgets.HBox([comparison_alpha_chain, comparison_beta_chain]),
    comparison_name,
]))

## 5. Run `per-residue-structural-variance.R`

In [ ]:
import os
import subprocess
import textwrap

BASE_PDB = BASE_PDB_FILE
COMPARISON_PDB = COMPARISON_PDB_FILE
BASE_ALPHA_CHAIN = label_to_chain(base_alpha_chain.value)
BASE_BETA_CHAIN = label_to_chain(base_beta_chain.value)
COMPARISON_ALPHA_CHAIN = label_to_chain(comparison_alpha_chain.value)
COMPARISON_BETA_CHAIN = label_to_chain(comparison_beta_chain.value)
COMPARISON_NAME = comparison_name.value.strip() or "prsv-comparison"

OUTPUT_CSV = f"{COMPARISON_NAME}_prsv.csv"
OUTPUT_PNG = f"{COMPARISON_NAME}_prsv.png"

runner = r'''
source("per-residue-structural-variance.R")

base_path <- Sys.getenv("BASE_PDB")
comparison_path <- Sys.getenv("COMPARISON_PDB")
comparison_name <- Sys.getenv("COMPARISON_NAME")
output_csv <- Sys.getenv("OUTPUT_CSV")
output_png <- Sys.getenv("OUTPUT_PNG")

base <- bio3d::read.pdb(base_path)
comparison <- bio3d::read.pdb(comparison_path)

result <- prsv(base, comparison,
               base.alpha.chain = Sys.getenv("BASE_ALPHA_CHAIN"),
               base.beta.chain = Sys.getenv("BASE_BETA_CHAIN"),
               comp.alpha.chain = Sys.getenv("COMPARISON_ALPHA_CHAIN"),
               comp.beta.chain = Sys.getenv("COMPARISON_BETA_CHAIN")) %>%
  mutate(Comparison = comparison_name)
write.csv(result, output_csv, row.names = FALSE)

plot <- result %>%
  ggplot(aes(x = Res, y = PRSV, color = Tubulin)) +
  geom_line(linewidth = 0.6) +
  facet_wrap(~Tubulin, ncol = 1, scales = "free_x") +
  labs(x = "Residue", y = expression(bold(paste("Per Residue Structural Variance ", (ring(A)^2)))), title = comparison_name) +
  theme_pubr() +
  theme(plot.title = element_text(face = "bold"))

ggsave(output_png, plot, width = 7, height = 5, dpi = 300)
print(head(result))
message("Wrote ", output_csv)
message("Wrote ", output_png)
'''

with open("run_prsv_colab.R", "w") as handle:
    handle.write(textwrap.dedent(runner))

env = os.environ.copy()
env.update({
    "BASE_PDB": BASE_PDB,
    "COMPARISON_PDB": COMPARISON_PDB,
    "BASE_ALPHA_CHAIN": BASE_ALPHA_CHAIN,
    "BASE_BETA_CHAIN": BASE_BETA_CHAIN,
    "COMPARISON_ALPHA_CHAIN": COMPARISON_ALPHA_CHAIN,
    "COMPARISON_BETA_CHAIN": COMPARISON_BETA_CHAIN,
    "COMPARISON_NAME": COMPARISON_NAME,
    "OUTPUT_CSV": OUTPUT_CSV,
    "OUTPUT_PNG": OUTPUT_PNG,
})

subprocess.run(["Rscript", "run_prsv_colab.R"], check=True, env=env)

## 6. Preview and download outputs

In [ ]:
import pandas as pd
from IPython.display import Image, display
from google.colab import files

display(pd.read_csv(OUTPUT_CSV).head())
display(Image(filename=OUTPUT_PNG))

files.download(OUTPUT_CSV)
files.download(OUTPUT_PNG)